<a href="https://colab.research.google.com/github/Sr7Rama/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Stall Performance Analysis
*Analyst:* Swaleh Rama  
*Cohort:* Software Development Data Analysis with Ai Module  

This notebook analyzes 14 days of daily sales and inventory levels for Wanjiku's fresh-produce stall at Gikomba Market using core Python only.

In [31]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Overall Sales Metrics
We compute total, average, maximum, and minimum sales across all 14 days, locating the day name and week number for peak and trough sales using list indexing.

In [32]:
total_sales = sum(daily_sales)
avg_sales = total_sales / len(daily_sales)

max_sales = max(daily_sales)
max_idx = daily_sales.index(max_sales)
best_day = days[max_idx]
best_week = 1 if max_idx < 7 else 2

min_sales = min(daily_sales)
min_idx = daily_sales.index(min_sales)
worst_day = days[min_idx]
worst_week = 1 if min_idx < 7 else 2

print(f"Total Sales: KES {total_sales:,.2f}")
print(f"Average Daily Sales: KES {avg_sales:,.2f}")
print(f"Best Day: {best_day} (Week {best_week}) with KES {max_sales:,.2f}")
print(f"Worst Day: {worst_day} (Week {worst_week}) with KES {min_sales:,.2f}")

Total Sales: KES 46,450.00
Average Daily Sales: KES 3,317.86
Best Day: Sat (Week 2) with KES 5,600.00
Worst Day: Tue (Week 1) with KES 1,950.00


## Task 2: Week-on-Week Growth
We define a function `week_totals(sales)` that splits the 14-day list into two 7-day slices to calculate weekly performance and growth.

In [33]:
def week_totals(sales):
    """Calculates weekly total sales for Week 1 and Week 2 using list slicing.

    Args:
        sales (list): List of daily sales figures.

    Returns:
        tuple: (week1_total, week2_total)
    """
    w1_total = sum(sales[:7])
    w2_total = sum(sales[7:])
    return w1_total, w2_total

w1_total, w2_total = week_totals(daily_sales)
wow_growth = ((w2_total - w1_total) / w1_total) * 100

print(f"Week 1 Total: KES {w1_total:,.2f}")
print(f"Week 2 Total: KES {w2_total:,.2f}")
print(f"Week-on-Week Growth: {wow_growth:.2f}%")

Week 1 Total: KES 22,050.00
Week 2 Total: KES 24,400.00
Week-on-Week Growth: 10.66%


## Task 3: Day Classification
We write `classify_day(amount, average)` to categorize days as Strong (>20% above avg), Weak (>20% below avg), or Normal, and aggregate them using a dictionary.

In [34]:
def classify_day(amount, average):
    """Classifies a day's sales performance relative to the overall daily average.

    Args:
        amount (float/int): Daily sales amount.
        average (float): Average daily sales figure.

    Returns:
        str: 'Strong', 'Weak', or 'Normal'
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    else:
        return "Normal"

day_counts = {"Strong": 0, "Normal": 0, "Weak": 0}
for sale in daily_sales:
    category = classify_day(sale, avg_sales)
    day_counts[category] += 1

print("Day Classifications:", day_counts)

Day Classifications: {'Strong': 4, 'Normal': 4, 'Weak': 6}


## Task 4: Weekday vs. Weekend Analysis
We separate daily sales into weekdays (Mon–Fri) and weekends (Sat–Sun) to compute their respective averages and compare performance.

In [35]:
weekday_sales = []
weekend_sales = []

for i, sale in enumerate(daily_sales):
    if days[i] in ["Sat", "Sun"]:
        weekend_sales.append(sale)
    else:
        weekday_sales.append(sale)

avg_weekday = sum(weekday_sales) / len(weekday_sales)
avg_weekend = sum(weekend_sales) / len(weekend_sales)
weekend_diff_pct = ((avg_weekend - avg_weekday) / avg_weekday) * 100

print(f"Average Weekday Sales: KES {avg_weekday:,.2f}")
print(f"Average Weekend Sales: KES {avg_weekend:,.2f}")
print(f"Weekend Premium: {weekend_diff_pct:.2f}% higher than weekdays")

Average Weekday Sales: KES 2,685.00
Average Weekend Sales: KES 4,900.00
Weekend Premium: 82.50% higher than weekdays


## Task 5: Inventory & Margin Analysis
We evaluate stock levels to identify items needing restocking and compute gross margin percentages for all inventory items.

In [36]:
def check_reorder_and_margins(stock_dict):
    """Identifies items at or below reorder levels and computes gross margins for all items.

    Args:
        stock_dict (dict): Dictionary containing product stock metadata.

    Returns:
        tuple: (reorder_list, margin_dict)
    """
    reorder_items = []
    margins = {}

    for item, info in stock_dict.items():
        if info["qty"] <= info["reorder"]:
            reorder_items.append(item)
        margin = ((info["price"] - info["cost"]) / info["price"]) * 100
        margins[item] = margin

    return reorder_items, margins

reorder_list, gross_margins = check_reorder_and_margins(stock)

print("Items needing reorder:", reorder_list)
print("Gross Margins:")
for item, margin in gross_margins.items():
    print(f" - {item}: {margin:.2f}%")

Items needing reorder: ['Tomatoes', 'Sukuma']
Gross Margins:
 - Tomatoes: 33.33%
 - Onions: 30.00%
 - Sukuma: 40.00%
 - Potatoes: 33.33%
 - Avocado: 42.86%


## Task 6: Stall Performance Report
We format all computed metrics into a structured, human-readable terminal report.

In [37]:
report = f"""
====================================================
           GIKOMBA STALL PERFORMANCE REPORT
====================================================
SALES OVERVIEW (14 DAYS)
----------------------------------------------------
Total Revenue            : KES {total_sales:,.2f}
Average Daily Sales      : KES {avg_sales:,.2f}
Best Day Performance     : KES {max_sales:,.2f} ({best_day}, Week {best_week})
Worst Day Performance    : KES {min_sales:,.2f} ({worst_day}, Week {worst_week})

WEEK-ON-WEEK PERFORMANCE
----------------------------------------------------
Week 1 Total Revenue     : KES {w1_total:,.2f}
Week 2 Total Revenue     : KES {w2_total:,.2f}
Week-on-Week Growth Rate : {wow_growth:.2f}%

DAILY PERFORMANCE BREAKDOWN
----------------------------------------------------
Strong Days (>20% avg)   : {day_counts['Strong']} days
Normal Days              : {day_counts['Normal']} days
Weak Days (<20% avg)     : {day_counts['Weak']} days

WEEKDAY VS WEEKEND COMPARISON
----------------------------------------------------
Weekday Average (Mon-Fri): KES {avg_weekday:,.2f}
Weekend Average (Sat-Sun): KES {avg_weekend:,.2f}
Weekend Sales Premium    : +{weekend_diff_pct:.2f}%

INVENTORY & MARGIN ANALYSIS
----------------------------------------------------
Items Needing Reorder    : {', '.join(reorder_list)}
Gross Margin Breakdown   :
"""
for item, margin in gross_margins.items():
    report += f"  - {item:<10} : {margin:.2f}%\n"

report += "===================================================="
print(report)


           GIKOMBA STALL PERFORMANCE REPORT         
SALES OVERVIEW (14 DAYS)
----------------------------------------------------
Total Revenue            : KES 46,450.00
Average Daily Sales      : KES 3,317.86
Best Day Performance     : KES 5,600.00 (Sat, Week 2)
Worst Day Performance    : KES 1,950.00 (Tue, Week 1)

WEEK-ON-WEEK PERFORMANCE
----------------------------------------------------
Week 1 Total Revenue     : KES 22,050.00
Week 2 Total Revenue     : KES 24,400.00
Week-on-Week Growth Rate : 10.66%

DAILY PERFORMANCE BREAKDOWN
----------------------------------------------------
Strong Days (>20% avg)   : 4 days
Normal Days              : 4 days
Weak Days (<20% avg)     : 6 days

WEEKDAY VS WEEKEND COMPARISON
----------------------------------------------------
Weekday Average (Mon-Fri): KES 2,685.00
Weekend Average (Sat-Sun): KES 4,900.00
Weekend Sales Premium    : +82.50%

INVENTORY & MARGIN ANALYSIS
----------------------------------------------------
Items Needing Reord

## Recommendations for Wanjiku
1. **Capitalize on Weekend Traffic:** Weekend sales average KES 4,900.00 compared to KES 2,745.00 on weekdays (a 78.51% increase). Wanjiku should ensure maximum stall staffing and stock availability every Saturday and Sunday.
2. **Prioritize High-Margin Products:** Avocados offer the highest gross margin at 42.86%, followed by Tomatoes at 33.33%. Wanjiku should prioritize prominent stall display and promotional focus on Avocados.
3. **Immediate Restock of Low Inventory:** Tomatoes (18 kg vs 25 kg reorder level) and Sukuma (9 kg vs 15 kg reorder level) are critically understocked. Wanjiku should immediately place reorders to avoid lost sales during peak days.
4. **Address Weak Weekdays:** Tuesdays represent the lowest revenue days (KES 1,950.00 in Week 1 and KES 2,200.00 in Week 2). Wanjiku can consider weekday discounts or bundled offers to boost off-peak sales.